# Stage 3 — Data Prep & Preprocessing

Goal: turn the raw Mahabharata dump (mixed `.txt` / `.csv`) into one normalized UTF-8 corpus at `data/processed/corpus.txt`.

Steps:
1. Load every text-ish file under `data/raw/`.
2. Strip Project-Gutenberg style headers/footers.
3. Normalize whitespace, fix curly quotes, collapse blank lines.
4. Write a single concatenated corpus + a tiny train/val split.

In [1]:
import re
import unicodedata
from pathlib import Path
import pandas as pd

raw_dir = Path('data/raw')
proc_dir = Path('data/processed')
proc_dir.mkdir(parents=True, exist_ok=True)

assert raw_dir.exists() and any(raw_dir.iterdir()), 'Run 02_data_collection.ipynb first.'

In [2]:
def read_text(p: Path) -> str:
    if p.suffix.lower() == '.csv':
        df = pd.read_csv(p)
        # Concatenate every string column row-wise.
        text_cols = [c for c in df.columns if df[c].dtype == object]
        rows = df[text_cols].astype(str).agg(' '.join, axis=1)
        return '\n'.join(rows.tolist())
    return p.read_text(encoding='utf-8', errors='replace')

docs = {}
for p in sorted(raw_dir.iterdir()):
    if p.suffix.lower() in {'.txt', '.csv'}:
        docs[p.name] = read_text(p)
        print(f'{p.name:40s}  {len(docs[p.name]):>10,d} chars')
print(f'\nTotal raw chars: {sum(len(v) for v in docs.values()):,}')

1-18 books combined.txt                    1,701,795 chars
test_relations.csv                                20 chars

Total raw chars: 1,701,815


In [3]:
GUTENBERG_START = re.compile(r'\*\*\* ?START OF.*?\*\*\*', re.IGNORECASE)
GUTENBERG_END   = re.compile(r'\*\*\* ?END OF.*?\*\*\*',   re.IGNORECASE)
MULTI_BLANK     = re.compile(r'\n{3,}')
TRAIL_WS        = re.compile(r'[ \t]+(\n|$)')

def clean(text: str) -> str:
    text = unicodedata.normalize('NFKC', text)
    # Drop everything outside the Gutenberg markers when present.
    m_start = GUTENBERG_START.search(text)
    if m_start:
        text = text[m_start.end():]
    m_end = GUTENBERG_END.search(text)
    if m_end:
        text = text[:m_end.start()]
    # Curly quotes / dashes → ASCII.
    text = (text
            .replace('\u2018', "'").replace('\u2019', "'")
            .replace('\u201c', '"').replace('\u201d', '"')
            .replace('\u2013', '-').replace('\u2014', '-'))
    text = TRAIL_WS.sub(r'\1', text)
    text = MULTI_BLANK.sub('\n\n', text)
    return text.strip()

cleaned = {name: clean(t) for name, t in docs.items()}
for name, t in cleaned.items():
    print(f'{name:40s}  {len(t):>10,d} chars')

1-18 books combined.txt                    1,701,170 chars
test_relations.csv                                 0 chars


In [4]:
# One corpus, separator marks doc boundaries.
SEP = '\n\n<|endofdoc|>\n\n'
corpus = SEP.join(cleaned.values())

corpus_path = proc_dir / 'corpus.txt'
corpus_path.write_text(corpus, encoding='utf-8')
print(f'Wrote {corpus_path}  ({len(corpus):,} chars, {len(corpus.split()):,} whitespace tokens)')

Wrote data/processed/corpus.txt  (1,701,186 chars, 290,923 whitespace tokens)


In [5]:
from collections import Counter

print('First 400 chars:\n')
print(corpus[:400])
print('\n---\nVocab sanity (unique chars):', len(set(corpus)))
print('Top 20 chars:', Counter(corpus).most_common(20))

First 400 chars:

Adi Parva

Chapter One
Maharaja Shantanu Marries the Celestial Ganga

According to the historical records of this earth, there once lived a King named Maharaja Shantanu, the son of Pratipa, who took his birth in the solar dynasty and was considered naradeva, the manifest representative of the Supreme Lord on earth. His fame and rule extended to all parts of the world. The qualities of self-control

---
Vocab sanity (unique chars): 84
Top 20 chars: [(' ', 288929), ('e', 156542), ('a', 130600), ('t', 108908), ('o', 93321), ('n', 91384), ('i', 90291), ('h', 89675), ('r', 88036), ('s', 84355), ('d', 60465), ('l', 48122), ('u', 39408), ('f', 30707), ('m', 29609), ('c', 28382), ('w', 26541), ('y', 24694), ('g', 24062), (',', 21202)]


In [6]:
# Simple 90/10 character split — fine for a from-scratch demo.
n = len(corpus)
split = int(0.9 * n)
(proc_dir / 'train.txt').write_text(corpus[:split], encoding='utf-8')
(proc_dir / 'val.txt').write_text(corpus[split:],   encoding='utf-8')
print(f'train: {split:,} chars   val: {n - split:,} chars')

train: 1,531,067 chars   val: 170,119 chars


Next: `04_tokenization.ipynb` — encode the corpus with a BPE tokenizer.